# 03 Gold Leakage Audit

This notebook audits the strict Gold leakage report. The model should select
features from the whitelist contract, not by dropping a small blacklist.

In [1]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

NOTEBOOK_NAME = "03_gold_leakage_audit"
GOLD = ROOT / "data" / "gold"
FEATURES = GOLD / "features"
LABELS = GOLD / "labels"
TRAINING = GOLD / "training"
METADATA = GOLD / "metadata"

OUTPUT_TABLES = ROOT / "eda" / "gold" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "gold" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "gold" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "gold" / "insights"
CHECKPOINTS = ROOT / "eda" / "gold" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def save_chart(fig, name: str) -> None:
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")

print("=" * 72)
print(f"GOLD EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Gold root: {GOLD}")


GOLD EDA - 03_gold_leakage_audit
Start time: 2026-06-02 14:28:40.732308
Gold root: D:\F1_WinRate_Predictor\data\gold


## Leakage Gate

The leakage gate blocks post-race outcome fields, duplicate feature keys,
duplicate label keys, nullable targets, and unregistered feature columns. A
clean report is required before the Gold training datasets are trusted.

In [2]:
leakage = read_json(METADATA / "leakage_report.json")
checks = [
    ("banned_columns_in_features", len(leakage.get("banned_columns_in_features", []))),
    ("duplicate_feature_key_rows", int(leakage.get("duplicate_feature_key_rows", 0))),
    ("duplicate_label_key_rows", int(leakage.get("duplicate_label_key_rows", 0))),
    ("nullable_feature_keys", len(leakage.get("nullable_feature_keys", {}))),
    ("nullable_targets", len(leakage.get("nullable_targets", {}))),
    ("unchecked_model_columns", len(leakage.get("unchecked_model_columns", []))),
]
leakage_matrix = pd.DataFrame(checks, columns=["check", "issue_count"])
leakage_matrix["status"] = np.where(leakage_matrix["issue_count"].eq(0), "PASS", "FAIL")
leakage_matrix.to_csv(OUTPUT_TABLES / "leakage_quality_matrix.csv", index=False)

fig = px.bar(
    leakage_matrix,
    x="check",
    y="issue_count",
    color="status",
    title=f"Gold Leakage Gate: {leakage.get('status')}",
    labels={"check": "Leakage check", "issue_count": "Issue count"},
)
fig.update_layout(margin=dict(l=10, r=10, t=55, b=120))
save_chart(fig, "leakage_quality_matrix")
fig.show()

display(leakage_matrix)

,check,issue_count,status
0,banned_columns_in_features,0,PASS
1,duplicate_feature_key_rows,0,PASS
2,duplicate_label_key_rows,0,PASS
3,nullable_feature_keys,0,PASS
4,nullable_targets,0,PASS
5,unchecked_model_columns,0,PASS


## Feature Whitelist Coverage

The whitelist is the enforceable boundary between Gold and model training.
Every model-eligible column must be registered with an explicit availability
rule.

In [3]:
feature_contract = read_json(METADATA / "feature_contract.json")
whitelist_rows = []
for group_name, contract in feature_contract.get("feature_tables", {}).items():
    for column in contract.get("model_allowed_columns", []):
        whitelist_rows.append({
            "feature_group": group_name,
            "column": column,
            "availability_rule": contract.get("availability_rule"),
        })
whitelist = pd.DataFrame(whitelist_rows)
whitelist.to_csv(OUTPUT_TABLES / "model_feature_whitelist.csv", index=False)

group_summary = whitelist.groupby("feature_group").size().rename("allowed_columns").reset_index()
fig = px.bar(
    group_summary,
    x="feature_group",
    y="allowed_columns",
    text="allowed_columns",
    title="Model Feature Whitelist by Gold Feature Group",
    labels={"feature_group": "Feature group", "allowed_columns": "Allowed columns"},
)
fig.update_layout(margin=dict(l=10, r=10, t=55, b=80))
fig.update_traces(textposition="outside", cliponaxis=False)
save_chart(fig, "feature_whitelist_by_group")
fig.show()

display(group_summary)
display(whitelist.head(50))

,feature_group,allowed_columns
0,base,12
1,interval,6
2,overtake,5
3,pit,5
4,position,7
5,stint,7
6,weather,7


,feature_group,column,availability_rule
0,base,event_type,Uses only static session/driver context and cu...
1,base,year,Uses only static session/driver context and cu...
2,base,circuit_short_name,Uses only static session/driver context and cu...
3,base,country_name,Uses only static session/driver context and cu...
4,base,team_name,Uses only static session/driver context and cu...
5,base,country_code,Uses only static session/driver context and cu...
6,base,grid_position,Uses only static session/driver context and cu...
7,base,scheduled_laps,Uses only static session/driver context and cu...
8,base,race_progress_pct,Uses only static session/driver context and cu...
9,base,laps_to_go,Uses only static session/driver context and cu...


In [4]:
write_report("leakage_audit", {
    "status": leakage.get("status"),
    "blockers": leakage.get("blockers", []),
    "unchecked_model_columns": leakage.get("unchecked_model_columns", []),
    "whitelist_columns": int(len(whitelist)),
})
write_insight(
    "Gold Leakage Audit",
    [
        f"Leakage report status is {leakage.get('status')}.",
        f"The model whitelist contains {len(whitelist):,} explicitly registered columns.",
        "Label columns are not present in the feature artifact before training dataset generation.",
    ],
    leakage.get("blockers", []),
    [
        "Future model code should load this whitelist directly from feature_contract.json.",
        "Any new feature module must declare availability and model_allowed_columns before training use.",
    ],
)
(CHECKPOINTS / "gold_leakage_audit_completed.txt").write_text(datetime.now().isoformat(), encoding="utf-8")

26